Input x: [B, S, D]
    │
    ├──────────────────────────────────┐
    │                                  │ (residual)
    ▼                                  │
┌─────────────┐                        │
│  LayerNorm  │  ← Pre-norm            │
└─────────────┘                        │
    │                                  │
    ▼                                  │
┌─────────────┐                        │
│  Attention  │                        │
└─────────────┘                        │
    │                                  │
    ▼                                  │
   Add ◄───────────────────────────────┘
    │
    ├──────────────────────────────────┐
    │                                  │ (residual)
    ▼                                  │
┌─────────────┐                        │
│  LayerNorm  │  ← Pre-norm            │
└─────────────┘                        │
    │                                  │
    ▼                                  │
┌─────────────┐                        │
│    MLP      │                        │
└─────────────┘                        │
    │                                  │
    ▼                                  │
   Add ◄───────────────────────────────┘
    │
Output: [B, S, D]

In [ ]:
x = x + Attention(LayerNorm(x))
x = x + MLP(LayerNorm(x))

Why pre-norm over post norm in Modern LLMs?
Post Norm
• gradients must pass through the normalization operation on the main identity path during backpropagation.
• Vanishing/exploding gradients: As models grow very deep, these operations shrink or distort gradients in the lower layers, causing early layers to stop learning.
• Strict warmup requirements: Post-Norm forces trainers to rely on aggressive learning rate warmups and delicate hyperparameter tuning to keep the network from crashing at initialization

Pre Norm
• leaves a clean, direct residual path untouched.
• Smooth gradient flow: Gradients can flow straight from the output all the way to the input without being crushed by normalization layers.
• No heavy warmup needed: Models train stably right from step one, which is why architectures like GPT-2, GPT-3, LLaMA, and Falcon use Pre-Norm.



In [ ]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import math

class LayerNorm(nn.Module):

    def __init__(self,dim: int , eps : int = 1e-5):

        super().__init__()

        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(self.dim))
        self.bias = nn.Parameter(torch.zeros(self.dim))


    def forward (self,x):

        B,S,D = x.shape 

        mean_x = torch.mean(x,dim=-1,keepdim=True)

        var_x = torch.var(x,dim=-1,keepdim=True)

        x_norm = (x-mean_x)/ torch.sqrt(var_x + self.eps)

        return  self.weight*x_norm + self.bias

In [ ]:
# For Rope use an external library 
from torchtune.modules import RotaryPositionalEmbeddings

rope = RotaryPositionalEmbeddings(dim=head_dim,max_seq_len=4192)

# Apply to Q,K of shape [B,H,S,D_H]

q = rope(q)
k = rope(k)

In [ ]:
# Multi Head attention 
from torchtune.modules import RotaryPositionalEmbeddings

class MultiHeadAttention(nn.Module):

    def __init__(self,d_model : int, n_heads: int, max_len: int=4192):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads

        self.h_dim = self.d_model // self.n_heads

        self.qkv_linear = nn.Linear(d_model,3*d_model)
        self.out_proj = nn.Linear(d_model,d_model)

        self.rope = RotaryPositionalEmbeddings(dim=self.h_dim,max_seq_len=max_len)

        self.scale = 1 / (self.h_dim ** 0.5)

    def forward(self, x, mask,casual_mask: bool= False):

        B,S,D = x.shape 

        qkv = self.qkv_linear(x) # [B,S,3*D]

        q,k,v = qkv.chunk(3,dim=-1) # [B,S,D]

        q = q.view(B,S,self.n_heads,self.h_dim) # [B,S,H,D_H]
        k = k.view(B,S,self.n_heads,self.h_dim)
        v = v.view(B,S,self.n_heads,self.h_dim)

        q = rope(q) #  [B,H,S,D_H]
        k = rope(k)

        q = torch.permute(q,(0,2,1,3)) # [B,H,S,D_H]
        k = torch.permute(k,(0,2,1,3))
        v = torch.permute(v,(0,2,1,3))       

        attn_score = torch.matmul(q,torch.transpose(k,2,3))*self.scale # [B,H,S,S]

        if casual_mask:
            # Create a 2D causal mask [S_q, S_k]
            mask_val = -1e9  # Use a large negative number instead of float('-inf') for numerical stability
            causal_pattern = torch.full((S_q, S_k), float('-inf'), device=q.device)
            causal_pattern = torch.triu(causal_pattern, diagonal=1) # Upper triangle is -inf
            
            # Broadcast and add to scores
            attn_score = attn_score + causal_pattern

        if mask is not None :

            attn_score = attn_score.masked_fill(mask == 0 , float('-inf'))


        attn_weights = F.softmax(attn_score,dim=-1)

        out = torch.matmul(attn_weights,v) # [B,H,S,S] * [B,H,S,D_H] = [B,H,S,D_H]

        out = torch.permute(out,(0,2,1,3)) # [B,S,H,D_H]

        out = out.contiguous().view(B,S,D) # [B,S,D]

        return  self.out_proj(out)


In [ ]:
# MLP layer 
class MLP(nn.Module):

    def __init__(self, d_model: int, d_ff, dropout : int = 0.1):
        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff 
        self.dropout = nn.Dropout(dropout)

        self.f1 = nn.Linear(d_model,d_ff)
        self.f2 = nn.Linear(d_ff,d_model)


    def forward(self,x):

        x = self.f1(x)
        x = F.gelu(x)
        x = self.f2(x)
        x = self.dropout(x)

        return x

In [ ]:
# Transformer Block 

class Transformer(nn.Module):

    def __init__(self,d_model,d_ff,n_heads,dropout):

        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff 

        self.dropout = dropout
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

        self.mha = MultiHeadAttention(d_model,n_heads)

        self.mlp = MLP(d_model,d_ff)


    def forward(self,x,mask):

        x = x + self.mha(self.norm1(x),mask)
        x = x + self.mlp(self.norm2(x))

        return x 

